# SCF Phase 2 shard 38

Sweeps: robustness. Jobs: 13. Projected: 5.0 h
(safety-factor 1.8x applied). Grids hash: `68970a975545`.
Code source: github.com/hugogobato/scf-confounding-frontier @ tag `phase2-freeze` (pinned for
reproducibility).
Pre-registration: `docs/phase2_preregistration.md` (thresholds frozen before
any data generation; deviation register D1-D7 included there).

Resume-safe: completed cells are skipped on rerun (checkpoint parquet per
cell). If the notebook approaches the Colab wall limit it finishes the
current cell and stops cleanly; rerun to continue.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"
!pip install -q "numpy>=2.0" "scipy>=1.14" "pandas>=2.2" "pyarrow>=16" scikit-learn

In [ ]:
!git clone --depth 1 --branch phase2-freeze \
    https://github.com/hugogobato/scf-confounding-frontier.git scf_repo
import sys, hashlib, json
sys.path.insert(0, "scf_repo/code")
# verify the pinned code matches the manifest recorded at generation time
EXPECTED = json.loads("{\"de_formulas.py\": \"5dffb441b638\", \"simulator.py\": \"ef31ca2a201b\", \"estimators.py\": \"7e27f25b2330\", \"detection.py\": \"06586fe60b9f\", \"runners.py\": \"df67486b60f5\"}")
for fname, short in EXPECTED.items():
    h = hashlib.sha256(open(f"scf_repo/code/{fname}", "rb").read()).hexdigest()[:12]
    assert h == short, f"code mismatch: {fname} ({h} != {short})"
print("code verified against generation-time hashes")

In [ ]:
import json, time, traceback
from multiprocessing import Pool
from runners import run_cell

JOBS = json.loads("[{\"config\": {\"n\": 2000, \"p\": 4000, \"r\": 5, \"l\": [0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"V6_sparse_conf\", \"conf_kind\": \"sparse\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"42fd97cca262\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/42fd97cca262.parquet\", \"means_path\": \"data/sim/robustness/means/42fd97cca262.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 4000, \"r\": 5, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"V6_sparse_conf\", \"conf_kind\": \"sparse\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"6bfa750d8d97\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/6bfa750d8d97.parquet\", \"means_path\": \"data/sim/robustness/means/6bfa750d8d97.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 5, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"V5_r-1\", \"r_misspec\": -1, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"11b8f6b9f5d0\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 125, \"raw_path\": \"data/sim/robustness/raw/11b8f6b9f5d0.parquet\", \"means_path\": \"data/sim/robustness/means/11b8f6b9f5d0.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 5, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"V5_r+1\", \"r_misspec\": 1, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"52875dc28af6\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 125, \"raw_path\": \"data/sim/robustness/raw/52875dc28af6.parquet\", \"means_path\": \"data/sim/robustness/means/52875dc28af6.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"V0_gauss\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"577631042004\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/577631042004.parquet\", \"means_path\": \"data/sim/robustness/means/577631042004.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"V0_gauss\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"60913fac11bd\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/60913fac11bd.parquet\", \"means_path\": \"data/sim/robustness/means/60913fac11bd.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"V1_t5\", \"error_law\": \"t5\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"a0304ef84687\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/a0304ef84687.parquet\", \"means_path\": \"data/sim/robustness/means/a0304ef84687.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"V1_t5\", \"error_law\": \"t5\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"caa086de4c4a\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/caa086de4c4a.parquet\", \"means_path\": \"data/sim/robustness/means/caa086de4c4a.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"V2_rademacher_half\", \"loading_kind\": \"rademacher_half\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"6a7c5efbe32a\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/6a7c5efbe32a.parquet\", \"means_path\": \"data/sim/robustness/means/6a7c5efbe32a.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"V2_rademacher_half\", \"loading_kind\": \"rademacher_half\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"799c7094aeb7\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/799c7094aeb7.parquet\", \"means_path\": \"data/sim/robustness/means/799c7094aeb7.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"V3_hetero_u\", \"hetero_u\": true, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"7de56325a429\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/7de56325a429.parquet\", \"means_path\": \"data/sim/robustness/means/7de56325a429.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"V3_hetero_u\", \"hetero_u\": true, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"2a6a9e144eb7\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/2a6a9e144eb7.parquet\", \"means_path\": \"data/sim/robustness/means/2a6a9e144eb7.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"V4_corr_f\", \"corr_factors\": true, \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"53f9e4049a7f\", \"mode\": \"robustness\", \"sweep\": \"robustness\", \"reps\": 250, \"raw_path\": \"data/sim/robustness/raw/53f9e4049a7f.parquet\", \"means_path\": \"data/sim/robustness/means/53f9e4049a7f.npz\", \"_rank\": 6, \"_sweep\": \"robustness\"}]")

def _safe(job):
    try:
        return run_cell(job)
    except Exception as e:
        print('[FAIL]', job['config_id'], repr(e))
        traceback.print_exc()
        return job['config_id'], -1.0

t0 = time.time()
results = []
for i, job in enumerate(JOBS):
    if time.time() - t0 > 8.6 * 3600:
        print('[WALL LIMIT] stopping cleanly after', i, 'jobs')
        break
    results.append(_safe(job))
print('shard done:', results)

In [ ]:
import hashlib, json, glob, os
manifest = {'shard_id': 38, 'files': {}}
os.makedirs('data', exist_ok=True)
for f in sorted(glob.glob('data/**/*.parquet', recursive=True)) + \
         sorted(glob.glob('data/**/*.npz', recursive=True)):
    h = hashlib.sha256(open(f, 'rb').read()).hexdigest()
    manifest['files'][f] = h
with open('data/manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=1)
print(json.dumps(manifest['files'], indent=1))

In [ ]:
import shutil
archive = shutil.make_archive('scf_shard_{:02d}'.format(38), 'zip', 'data')
print('archived:', archive)
output_file = archive
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)